In [1]:
import numpy as np
import pandas as pd
import json

np.random.seed(33)

# Sumber 1: Transaksi (CSV) — 5000 baris
n_trx = 5000
kategori_list = ["Elektronik", "Fashion", "Makanan", "Rumah Tangga", "Kesehatan"]
df_trx_tugas6 = pd.DataFrame({
    "order_id": [f"TX{i}" for i in range(n_trx)],
    "product_id": np.random.randint(1, 31, size=n_trx),
    "unit_terjual": np.random.randint(1, 8, size=n_trx),
    "tanggal": np.random.choice(pd.date_range("2026-10-01","2026-10-31"), size=n_trx).astype(str),
})
df_trx_tugas6.to_csv("tugas6_transaksi.csv", index=False)

# Sumber 2: Data produk (JSON Lines) — 30 produk
produk = [
    {"product_id": i, "nama_produk": f"Produk-{i}", "kategori": np.random.choice(kategori_list),
     "harga": int(np.random.choice([25000,50000,75000,100000,150000,250000]))}
    for i in range(1, 31)
]
with open("tugas6_produk.json", "w") as f:
    for p in produk:
        f.write(json.dumps(p) + "\n")

# Sumber 3: Data ulasan (CSV) — tidak semua transaksi memiliki ulasan (realistis)
n_review = 3500
df_review = pd.DataFrame({
    "order_id": np.random.choice(df_trx_tugas6["order_id"], size=n_review, replace=False),
    "rating": np.random.randint(1, 6, size=n_review),
})
df_review.to_csv("tugas6_ulasan.csv", index=False)

print(f"Tiga sumber data berhasil dibuat:")
print(f"- tugas6_transaksi.csv : {len(df_trx_tugas6)} baris")
print(f"- tugas6_produk.json   : {len(produk)} baris")
print(f"- tugas6_ulasan.csv    : {len(df_review)} baris (tidak seluruh transaksi memiliki ulasan)")

Tiga sumber data berhasil dibuat:
- tugas6_transaksi.csv : 5000 baris
- tugas6_produk.json   : 30 baris
- tugas6_ulasan.csv    : 3500 baris (tidak seluruh transaksi memiliki ulasan)


In [3]:
from pyspark.sql import SparkSession

# Inisialisasi SparkSession
spark = SparkSession.builder \
    .appName("Tugas6_ETL_Pipeline") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

print("SparkSession berhasil diinisialisasi!")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/24 03:20:10 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/09/24 03:20:12 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


SparkSession berhasil diinisialisasi!


In [4]:
#A. EXTRACT
print("=== TAHAP EXTRACT ===")
df_trx = spark.read.csv("tugas6_transaksi.csv", header=True, inferSchema=True)
df_prod = spark.read.json("tugas6_produk.json")
df_rev = spark.read.csv("tugas6_ulasan.csv", header=True, inferSchema=True)

print(f"Jumlah baris Transaksi : {df_trx.count()}")
df_trx.printSchema()

print(f"Jumlah baris Produk      : {df_prod.count()}")
df_prod.printSchema()

print(f"Jumlah baris Ulasan      : {df_rev.count()}")
df_rev.printSchema()

=== TAHAP EXTRACT ===
Jumlah baris Transaksi : 5000
root
 |-- order_id: string (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- unit_terjual: integer (nullable = true)
 |-- tanggal: timestamp (nullable = true)

Jumlah baris Produk      : 30
root
 |-- harga: long (nullable = true)
 |-- kategori: string (nullable = true)
 |-- nama_produk: string (nullable = true)
 |-- product_id: long (nullable = true)

Jumlah baris Ulasan      : 3500
root
 |-- order_id: string (nullable = true)
 |-- rating: integer (nullable = true)



In [6]:
# B. TRANSFORM — Penggabungan & Kolom Turunan
from pyspark.sql.functions import col, when, lit, count, avg
print("=== TAHAP TRANSFORM (JOIN) ===")
# Inner join dari transaksi ke produk
df_joined = df_trx.join(df_prod, on="product_id", how="inner")

# Left join dari transaksi ke ulasan
df_joined = df_joined.join(df_rev, on="order_id", how="left")

# Menambahkan kolom turunan total_pendapatan (unit_terjual * harga)
df_joined = df_joined.withColumn("total_pendapatan", col("unit_terjual") * col("harga"))

df_joined.select("order_id", "product_id", "kategori", "unit_terjual", "harga", "total_pendapatan", "rating").show(5)

=== TAHAP TRANSFORM (JOIN) ===
+--------+----------+------------+------------+------+----------------+------+
|order_id|product_id|    kategori|unit_terjual| harga|total_pendapatan|rating|
+--------+----------+------------+------------+------+----------------+------+
|     TX0|        21|  Elektronik|           2| 25000|           50000|     1|
|     TX1|         8|     Fashion|           6|250000|         1500000|  NULL|
|     TX2|        25|  Elektronik|           4| 25000|          100000|     2|
|     TX3|         3|     Fashion|           4| 75000|          300000|     5|
|     TX4|        19|Rumah Tangga|           6| 75000|          450000|  NULL|
+--------+----------+------------+------------+------+----------------+------+
only showing top 5 rows



In [7]:
# C. TRANSFORM — Penanganan Data Kosong & Pengayaan
print("=== TAHAP TRANSFORM (DATA QUALITY) ===")
# Membuat kolom penanda ada_ulasan SEBELUM na.fill()
df_transformed = df_joined.withColumn(
    "ada_ulasan",
    when(col("rating").isNotNull(), True).otherwise(False)
)

# Mengisi nilai kosong pada kolom rating dengan angka 0
df_transformed = df_transformed.na.fill({"rating": 0})

df_transformed.select("order_id", "rating", "ada_ulasan", "total_pendapatan").show(5)

=== TAHAP TRANSFORM (DATA QUALITY) ===
+--------+------+----------+----------------+
|order_id|rating|ada_ulasan|total_pendapatan|
+--------+------+----------+----------------+
|     TX0|     1|      true|           50000|
|     TX1|     0|     false|         1500000|
|     TX2|     2|      true|          100000|
|     TX3|     5|      true|          300000|
|     TX4|     0|     false|          450000|
+--------+------+----------+----------------+
only showing top 5 rows



In [8]:
# D. LOAD
print("=== TAHAP LOAD KE HDFS ===")
import subprocess
subprocess.run(["hdfs", "dfs", "-mkdir", "-p", "/user/nico/tugas6"])

output_path = "hdfs://localhost:9000/user/nico/tugas6/hasil_etl"
df_transformed.write.mode("overwrite").partitionBy("kategori").parquet(output_path)

print("Pipeline ETL selesai — data berhasil dimuat ke HDFS.")

# Verifikasi struktur direktori HDFS
!hdfs dfs -ls -R /user/nico/tugas6/hasil_etl

# Membaca kembali dari HDFS untuk membuktikan data utuh
df_verified = spark.read.parquet(output_path)
print(f"Jumlah baris terverifikasi di HDFS: {df_verified.count()}")

=== TAHAP LOAD KE HDFS ===


Pipeline ETL selesai — data berhasil dimuat ke HDFS.
-rw-r--r--   3 nico supergroup          0 2026-09-24 03:22 /user/nico/tugas6/hasil_etl/_SUCCESS
drwxr-xr-x   - nico supergroup          0 2026-09-24 03:22 /user/nico/tugas6/hasil_etl/kategori=Elektronik
-rw-r--r--   3 nico supergroup      16361 2026-09-24 03:22 /user/nico/tugas6/hasil_etl/kategori=Elektronik/part-00000-711cfeff-ffb8-437b-85bb-add2824732b3.c000.snappy.parquet
drwxr-xr-x   - nico supergroup          0 2026-09-24 03:22 /user/nico/tugas6/hasil_etl/kategori=Fashion
-rw-r--r--   3 nico supergroup      11131 2026-09-24 03:22 /user/nico/tugas6/hasil_etl/kategori=Fashion/part-00000-711cfeff-ffb8-437b-85bb-add2824732b3.c000.snappy.parquet
drwxr-xr-x   - nico supergroup          0 2026-09-24 03:22 /user/nico/tugas6/hasil_etl/kategori=Kesehatan
-rw-r--r--   3 nico supergroup      10389 2026-09-24 03:22 /user/nico/tugas6/hasil_etl/kategori=Kesehatan/part-00000-711cfeff-ffb8-437b-85bb-add2824732b3.c000.snappy.parquet
drwxr-xr-x   

In [9]:
# E. Insight Akhir
print("=== E. INSIGHT AKHIR ===")
df_insight = df_verified.groupBy("kategori").agg(
    count("order_id").alias("total_transaksi"),
    avg(when(col("ada_ulasan") == True, 1).otherwise(0)).alias("rasio_ulasan")
).orderBy("rasio_ulasan", ascending=True)

df_insight.show()

=== E. INSIGHT AKHIR ===
+------------+---------------+------------------+
|    kategori|total_transaksi|      rasio_ulasan|
+------------+---------------+------------------+
|     Makanan|            536|0.6884328358208955|
|   Kesehatan|            961|0.6888657648283039|
|     Fashion|           1035|0.7014492753623188|
|Rumah Tangga|            815|0.7055214723926381|
|  Elektronik|           1653|0.7065940713853599|
+------------+---------------+------------------+



Berdasarkan hasil analisis ETL di atas, kategori produk yang memiliki persentase ulasan paling rendah mengindikasikan rendahnya keterlibatan (engagement) pembeli untuk memberikan ulasan secara sukarela. Informasi ini sangat penting bagi tim marketing untuk merancang strategi pendorong ulasan (seperti pemberian insentif poin atau kupon diskon khusus) guna meningkatkan kepercayaan calon pembeli baru pada kategori produk tersebut.